# NPZ File Structure Analysis, PPG Visualization, Training and Inference

This notebook analyzes NPZ files from `/home/cristic/preprocessed_data/`, visualizes PPG data, trains an SCNN model on 9 ROIs, and performs inference.

**Features:**
- Filter and analyze only files with exactly 450 frames
- Train SCNN model on 9 ROIs with temporal consistency loss
- Inference on NPZ files and video files
- Visualization of predictions vs ground truth

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import json
import glob
from scipy import signal
from scipy.signal import butter, filtfilt, detrend, find_peaks

# PyTorch imports
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configuration
PREPROCESSED_DATA_PATH = Path('/home/cristic/preprocessed_data/')
EXPECTED_FRAMES = 450  # Only analyze files with exactly 450 frames
VIDEO_DATA_PATH = Path('/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/video/')

# Ensure paths exist
if not PREPROCESSED_DATA_PATH.exists():
    print(f"Warning: {PREPROCESSED_DATA_PATH} does not exist. Using current directory instead.")
    PREPROCESSED_DATA_PATH = Path('.')

# Check CUDA availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# ========================================================================
# MODEL DEFINITIONS
# ========================================================================

class SCNN_9rois(nn.Module):
    def __init__(self):
        super(SCNN_9rois, self).__init__()
        # Input channels: 9 ROIs * 3 channels (RGB) = 27 channels
        # Temporal depth fixed at 450 frames, spatial size 24x24
        self.spatial_temporal_feature_extractor = nn.Sequential(
            nn.Conv3d(in_channels=27, out_channels=32, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.Conv3d(32, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            # Pool spatial dimensions (H, W) down to 1x1 while keeping the 450 frames intact
            nn.AdaptiveAvgPool3d((450, 1, 1))
        )
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x):
        # x shape: (Batch, 27, 450, 24, 24)
        features = self.spatial_temporal_feature_extractor(x)  # (Batch, 64, 450, 1, 1)
        features = features.squeeze(-1).squeeze(-1)            # (Batch, 64, 450)
        features = features.permute(0, 2, 1)                    # (Batch, 450, 64)
        output_signal = self.regression_head(features)          # (Batch, 450, 1)
        return output_signal.squeeze(-1)                        # (Batch, 450)

In [ ]:
# ========================================================================
# LOSS FUNCTION
# ========================================================================

class NegPearsonLoss(nn.Module):
    def __init__(self):
        super(NegPearsonLoss, self).__init__()

    def forward(self, predictions, targets):
        x_mut = predictions - torch.mean(predictions, dim=-1, keepdim=True)
        y_mut = targets - torch.mean(targets, dim=-1, keepdim=True)
        r_num = torch.sum(x_mut * y_mut, dim=-1)
        r_den = torch.sqrt(torch.sum(x_mut ** 2, dim=-1) * torch.sum(y_mut ** 2, dim=-1) + 1e-6)
        return 1.0 - torch.mean(r_num / r_den)

In [ ]:
# ========================================================================
# DATASET CLASSES
# ========================================================================

class ScalableRppgDataset(Dataset):
    def __init__(self, file_paths, expected_frames=450, cache_file=None):
        self.selected_rois = [
            'roi_chin', 'roi_chin_199', 'roi_forehead', 'roi_left_cheek_280', 
            'roi_left_eye', 'roi_mouth', 'roi_nose', 'roi_right_cheek_50', 'roi_right_eye'
        ]
        self.expected_frames = expected_frames
        
        # Performance Optimization: Cache verified files to avoid rescanning
        if cache_file and os.path.exists(cache_file):
            print(f"Loading cached file paths from {cache_file}...")
            with open(cache_file, "r") as f:
                self.valid_file_paths = [line.strip() for line in f.readlines() if line.strip()]
        else:
            print(f"Scanning {len(file_paths)} files...")
            self.valid_file_paths = []
            for path in file_paths:
                try:
                    with np.load(path, mmap_mode='r') as data:
                        if data[self.selected_rois[0]].shape[0] == self.expected_frames:
                            self.valid_file_paths.append(path)
                except Exception:
                    continue
            
            # Write cache
            if cache_file:
                with open(cache_file, "w") as f:
                    for path in self.valid_file_paths:
                        f.write(f"{path}\n")
        
        print(f"Dataset ready. Total valid samples: {len(self.valid_file_paths)}")

    def __len__(self):
        return len(self.valid_file_paths)

    def __getitem__(self, idx):
        with np.load(self.valid_file_paths[idx], mmap_mode='r') as data:
            rois_list = []
            for roi_name in self.selected_rois:
                roi_data = np.array(data[roi_name], dtype=np.float32) / 255.0
                rois_list.append(roi_data)
            
            stacked_rois = np.concatenate(rois_list, axis=-1)
            stacked_rois = np.transpose(stacked_rois, (3, 0, 1, 2))
            
            ppg = np.array(data['ppg_values'], dtype=np.float32)
            ppg = (ppg - np.mean(ppg)) / (np.std(ppg) + 1e-6)
        
        return torch.from_numpy(stacked_rois), torch.from_numpy(ppg)

In [ ]:
# ========================================================================
# UTILITY FUNCTIONS
# ========================================================================

def load_npz_file(file_path):
    """Load and return the contents of an NPZ file."""
    try:
        data = np.load(file_path, allow_pickle=True)
        return data
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

def get_num_frames(file_path):
    """Get the number of frames from an NPZ file without loading all data."""
    try:
        data = np.load(file_path, allow_pickle=True)
        if 'ppg_values' in data:
            return data['ppg_values'].shape[0]
        elif 'meta_num_frames' in data:
            return int(data['meta_num_frames'])
        else:
            for key in data.files:
                if hasattr(data[key], 'shape') and len(data[key].shape) > 0:
                    return data[key].shape[0]
        return 0
    except Exception as e:
        print(f"Error getting frame count for {file_path}: {e}")
        return 0

def analyze_npz_structure(data):
    """Analyze and print the structure of an NPZ file."""
    print("=" * 60)
    print("NPZ File Structure Analysis")
    print("=" * 60)
    for key in sorted(data.files):
        item = data[key]
        if hasattr(item, 'shape'):
            print(f"  {key}: shape {item.shape}, dtype {item.dtype}")
        else:
            print(f"  {key}: {type(item).__name__} = {item}")
    print("=" * 60)
    return data

def extract_metadata(data):
    """Extract metadata from NPZ file."""
    metadata = {}
    for key in data.files:
        if key.startswith('meta_'):
            metadata[key] = data[key].item() if data[key].size == 1 else data[key]
        elif key.startswith('vital_'):
            metadata[key] = data[key].item() if data[key].size == 1 else data[key]
    return metadata

def extract_roi_names(data):
    """Extract ROI names from NPZ file."""
    rois = []
    for key in data.files:
        if key.startswith('roi_'):
            rois.append(key)
    return sorted(rois)

def compute_time_array(time_deltas, expected_fps=30):
    """Compute time array from time deltas, handling different time units."""
    time_array = np.cumsum(time_deltas)
    num_frames = len(time_deltas)
    total_time = time_array[-1]
    calculated_fps = num_frames / total_time if total_time > 0 else float('inf')
    
    if calculated_fps > 100:
        print(f"  Note: Time deltas appear to be in milliseconds (calculated FPS: {calculated_fps:.2f}). Converting to seconds.")
        time_array = time_array / 1000.0
        total_time = time_array[-1]
        calculated_fps = num_frames / total_time
        print(f"  After scaling: Total time = {total_time:.2f}s, FPS = {calculated_fps:.2f}")
    
    if expected_fps and abs(calculated_fps - expected_fps) > 5:
        scale_factor = num_frames / (expected_fps * total_time) if total_time > 0 else 1
        time_array = time_array * scale_factor
        total_time = time_array[-1]
        calculated_fps = num_frames / total_time
        print(f"  Adjusted to match expected FPS: Scale factor = {scale_factor:.4f}, New FPS = {calculated_fps:.2f}")
    
    return time_array

In [ ]:
# ========================================================================
# TRAINING PIPELINE
# ========================================================================

def train_pipeline(data_dir, epochs=10, batch_size=4, lr=1e-4):
    print(f"Active Execution Device: {device}")

    all_files = glob.glob(os.path.join(data_dir, "*.npz"))
    if not all_files:
        raise FileNotFoundError(f"Directory {data_dir} contains zero processing targets.")

    # Group files by subject identifiers for proper train/val split
    subject_pools = {}
    for f in all_files:
        try:
            meta = np.load(f)
            sub_id = int(meta['meta_subject_id'])
            subject_pools.setdefault(sub_id, []).append(f)
        except Exception:
            continue

    subjects = list(subject_pools.keys())
    split_border = int(len(subjects) * 0.8)
    train_subs, val_subs = subjects[:split_border], subjects[split_border:]
    
    train_files = [f for s in train_subs for f in subject_pools[s]]
    val_files = [f for s in val_subs for f in subject_pools[s]]

    print(f"Files allocated -> Training: {len(train_files)} | Validation: {len(val_files)}")

    # Create datasets with caching
    train_dataset = ScalableRppgDataset(
        file_paths=train_files, 
        cache_file=os.path.join(data_dir, "train_files_cache.txt")
    )
    val_dataset = ScalableRppgDataset(
        file_paths=val_files, 
        cache_file=os.path.join(data_dir, "val_files_cache.txt")
    )

    # Create data loaders
    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        drop_last=True
    )
    
    val_loader = DataLoader(
        dataset=val_dataset,
        batch_size=batch_size,
        shuffle=False, 
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        drop_last=False
    )

    # Initialize model, loss, optimizer
    model = SCNN_9rois().to(device)
    pearson_loss = NegPearsonLoss()
    mse_loss = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'val_rmse': []}
    best_val_loss = float('inf')
    checkpoint_path = os.path.join(data_dir, "best_scnn_9rois.pth")

    for epoch in range(epochs):
        model.train()
        running_train_loss = 0.0
        
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = pearson_loss(outputs, targets) + 0.5 * mse_loss(outputs, targets)
            loss.backward()
            optimizer.step()
            running_train_loss += loss.item() * inputs.size(0)

        epoch_train_loss = running_train_loss / len(train_loader.dataset)
        
        # Validation Pass
        model.eval()
        running_val_loss = 0.0
        absolute_error_sum = 0.0
        squared_error_sum = 0.0
        total_data_points = 0
        
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                v_loss = pearson_loss(outputs, targets) + 0.5 * mse_loss(outputs, targets)
                running_val_loss += v_loss.item() * inputs.size(0)
                absolute_error_sum += torch.sum(torch.abs(outputs - targets)).item()
                squared_error_sum += torch.sum((outputs - targets) ** 2).item()
                total_data_points += targets.numel()

        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        epoch_mae = absolute_error_sum / total_data_points
        epoch_rmse = np.sqrt(squared_error_sum / total_data_points)

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['val_mae'].append(epoch_mae)
        history['val_rmse'].append(epoch_rmse)

        # Save best model
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(), checkpoint_path)
            save_status = f" -> Saved new best weights to {checkpoint_path}"
        else:
            save_status = ""

        print(f"[Epoch {epoch+1:02d}/{epochs:02d}] \
              f"Train Loss: {epoch_train_loss:.4f} | \
              f"Val Loss: {epoch_val_loss:.4f} | \
              f"MAE: {epoch_mae:.4f} | \
              f"RMSE: {epoch_rmse:.4f}{save_status}")

    # Plot training history
    plot_evaluation_dashboard(history)
    
    return model, history

def plot_evaluation_dashboard(history):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss plot
    ax1.plot(epochs, history['train_loss'], '-o', label='Train Loss', color='#1f77b4', linewidth=2)
    ax1.plot(epochs, history['val_loss'], '-s', label='Validation Loss', color='#ff7f0e', linewidth=2)
    ax1.set_title('9-ROI Spatial-Temporal Loss Trajectory', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Epoch Iteration')
    ax1.set_ylabel('Loss Value')
    ax1.legend()
    ax1.grid(True, linestyle=':', alpha=0.6)
    
    # Error metrics plot
    ax2.plot(epochs, history['val_mae'], '-^', label='MAE Error', color='#2ca02c', linewidth=2)
    ax2.plot(epochs, history['val_rmse'], '-d', label='RMSE Error', color='#d62728', linewidth=2)
    ax2.set_title('Validation Error Tracking Metrics', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Epoch Iteration')
    ax2.set_ylabel('Amplitude Error Variance')
    ax2.legend()
    ax2.grid(True, linestyle=':', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ========================================================================
# INFERENCE FUNCTIONS
# ========================================================================

def run_single_file_inference(model_path, sample_npz_path, device_str='cuda'):
    """Run inference on a single NPZ file."""
    device = torch.device(device_str if torch.cuda.is_available() else 'cpu')
    
    # Initialize model
    model = SCNN_9rois()
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    # Selected ROIs
    selected_rois = [
        'roi_chin', 'roi_chin_199', 'roi_forehead', 'roi_left_cheek_280', 
        'roi_left_eye', 'roi_mouth', 'roi_nose', 'roi_right_cheek_50', 'roi_right_eye'
    ]
    
    # Load and preprocess data
    with np.load(sample_npz_path) as data:
        rois_list = []
        for roi_name in selected_rois:
            roi_data = data[roi_name].astype(np.float32) / 255.0
            rois_list.append(roi_data)
        
        stacked_rois = np.concatenate(rois_list, axis=-1)
        stacked_rois = np.transpose(stacked_rois, (3, 0, 1, 2))
        
        gt_ppg = data['ppg_values'].astype(np.float32)
        gt_ppg = (gt_ppg - np.mean(gt_ppg)) / (np.std(gt_ppg) + 1e-6)
        
        subject_id = data['meta_subject_id']
        condition = data['meta_condition']
        camera_type = data.get('meta_camera_type', 'N/A')

    # Model inference
    input_tensor = torch.tensor(stacked_rois, dtype=torch.float32).unsqueeze(0).to(device)
    
    with torch.no_grad():
        prediction = model(input_tensor)
        pred_ppg = prediction.squeeze(0).cpu().numpy()

    # Plot comparison
    plt.figure(figsize=(15, 5))
    time_axis = np.arange(len(gt_ppg))
    
    plt.plot(time_axis, gt_ppg, label='Ground Truth PPG', color='#2ca02c', linewidth=2, alpha=0.8)
    plt.plot(time_axis, pred_ppg, label='SCNN 9-ROI Prediction', color='#d62728', linewidth=2, linestyle='--')
    
    title = f"rPPG Waveform (Subject: {subject_id} | Condition: {condition} | Camera: {camera_type})"
    plt.title(title, fontsize=12, fontweight='bold')
    plt.xlabel('Frame Timeline')
    plt.ylabel('Normalized Amplitude')
    plt.legend(loc='upper right')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()
    
    # Calculate metrics
    from scipy.stats import pearsonr
    corr, _ = pearsonr(gt_ppg, pred_ppg)
    mae = np.mean(np.abs(gt_ppg - pred_ppg))
    rmse = np.sqrt(np.mean((gt_ppg - pred_ppg) ** 2))
    
    print(f"Inference Metrics:")
    print(f"  Pearson Correlation: {corr:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    
    return pred_ppg, gt_ppg

In [ ]:
# ========================================================================
# VIDEO PROCESSING FUNCTIONS
# ========================================================================

def extract_frames_from_video(video_path, num_frames=450):
    """Extract frames from video file."""
    try:
        import skvideo.io
        videodata = skvideo.io.vread(video_path)
        
        # If video has more frames than needed, sample uniformly
        if len(videodata) > num_frames:
            indices = np.linspace(0, len(videodata) - 1, num_frames, dtype=int)
            videodata = videodata[indices]
        elif len(videodata) < num_frames:
            # Pad with last frame if video is too short
            last_frame = videodata[-1:].repeat(num_frames - len(videodata), axis=0)
            videodata = np.concatenate([videodata, last_frame], axis=0)
        
        return videodata
    except ImportError:
        print("skvideo not available. Install with: pip install scikit-video")
        return None
    except Exception as e:
        print(f"Error reading video {video_path}: {e}")
        return None

def detect_face_and_extract_rois(frame, face_detector=None):
    """Detect face and extract ROI coordinates."""
    try:
        import mediapipe as mp
        if face_detector is None:
            face_detector = mp.solutions.face_mesh.FaceMesh(
                static_image_mode=True,
                max_num_faces=1,
                refine_landmarks=True,
                min_detection_confidence=0.5
            )
        
        # Convert to RGB
        rgb_frame = frame.copy()
        if len(rgb_frame.shape) == 2:
            rgb_frame = np.stack([rgb_frame] * 3, axis=-1)
        elif rgb_frame.shape[2] == 4:
            rgb_frame = rgb_frame[:, :, :3]
        
        results = face_detector.process(rgb_frame)
        
        if results.multi_face_landmarks:
            landmarks = results.multi_face_landmarks[0].landmark
            # Convert landmarks to pixel coordinates
            height, width = frame.shape[:2]
            landmark_points = np.array([
                (lmk.x * width, lmk.y * height) 
                for lmk in landmarks
            ])
            return landmark_points
        return None
    except ImportError:
        print("mediapipe not available. Install with: pip install mediapipe")
        return None
    except Exception as e:
        print(f"Error detecting face: {e}")
        return None

def extract_roi_from_frame(frame, landmarks, roi_indices, size=24):
    """Extract and resize a specific ROI from frame."""
    if landmarks is None or len(landmarks) == 0:
        return None
    
    # Get bounding box of ROI landmarks
    roi_landmarks = landmarks[roi_indices]
    x_coords = roi_landmarks[:, 0]
    y_coords = roi_landmarks[:, 1]
    
    x_min, x_max = int(x_coords.min()), int(x_coords.max())
    y_min, y_max = int(y_coords.min()), int(y_coords.max())
    
    # Add padding
    padding = 5
    x_min = max(0, x_min - padding)
    x_max = min(frame.shape[1], x_max + padding)
    y_min = max(0, y_min - padding)
    y_max = min(frame.shape[0], y_max + padding)
    
    # Extract ROI
    roi = frame[y_min:y_max, x_min:x_max]
    
    # Resize to target size
    from skimage.transform import resize
    roi = resize(roi, (size, size), mode='reflect', anti_aliasing=True)
    
    return roi

In [ ]:
# ROI definitions for mediapipe landmarks
ROI_DEFINITIONS = {
    'roi_full_face': list(range(468)),
    'roi_forehead': [103, 104, 105, 332, 333, 334, 6, 7, 8, 9, 10],
    'roi_left_eye': list(range(22, 53)),
    'roi_right_eye': list(range(252, 283)),
    'roi_nose': list(range(1, 21)) + list(range(195, 221)),
    'roi_mouth': list(range(60, 81)) + list(range(290, 321)),
    'roi_left_cheek': list(range(0, 101)) + list(range(200, 301)),
    'roi_right_cheek': list(range(100, 201)) + list(range(300, 401)),
    'roi_chin': list(range(150, 171)) + list(range(370, 391)),
    'roi_left_iris': list(range(468, 473)),
    'roi_right_iris': list(range(473, 478))
}

# For the 9 ROIs used in the model
MODEL_ROIS = [
    'roi_chin', 'roi_chin_199', 'roi_forehead', 'roi_left_cheek_280', 
    'roi_left_eye', 'roi_mouth', 'roi_nose', 'roi_right_cheek_50', 'roi_right_eye'
]

# Map model ROI names to landmark indices
MODEL_ROI_TO_INDICES = {
    'roi_chin': ROI_DEFINITIONS['roi_chin'],
    'roi_chin_199': ROI_DEFINITIONS['roi_chin'],  # Same as roi_chin for now
    'roi_forehead': ROI_DEFINITIONS['roi_forehead'],
    'roi_left_cheek_280': ROI_DEFINITIONS['roi_left_cheek'],
    'roi_left_eye': ROI_DEFINITIONS['roi_left_eye'],
    'roi_mouth': ROI_DEFINITIONS['roi_mouth'],
    'roi_nose': ROI_DEFINITIONS['roi_nose'],
    'roi_right_cheek_50': ROI_DEFINITIONS['roi_right_cheek'],
    'roi_right_eye': ROI_DEFINITIONS['roi_right_eye']
}

In [ ]:
def process_video_to_npz_format(video_path, output_npz_path, num_frames=450):
    """Process a video file to extract ROIs and save in NPZ format."""
    print(f"Processing video: {video_path}")
    
    # Extract frames
    frames = extract_frames_from_video(video_path, num_frames)
    if frames is None:
        print("Failed to extract frames from video")
        return None
    
    # Initialize face detector
    try:
        import mediapipe as mp
        face_detector = mp.solutions.face_mesh.FaceMesh(
            static_image_mode=True,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5
        )
    except ImportError:
        print("mediapipe not available")
        return None
    
    # Process each frame
    roi_data = {roi: [] for roi in MODEL_ROIS}
    all_landmarks = []
    
    for frame in frames:
        landmarks = detect_face_and_extract_rois(frame, face_detector)
        if landmarks is None:
            # If no face detected, use previous frame's landmarks or skip
            if all_landmarks:
                landmarks = all_landmarks[-1]
            else:
                print("No face detected in first frame")
                return None
        
        all_landmarks.append(landmarks.copy())
        
        # Extract each ROI
        for roi_name in MODEL_ROIS:
            roi_indices = MODEL_ROI_TO_INDICES[roi_name]
            roi = extract_roi_from_frame(frame, landmarks, roi_indices, size=24)
            if roi is not None:
                roi_data[roi_name].append(roi)
    
    # Convert to numpy arrays
    roi_arrays = {}
    for roi_name, roi_frames in roi_data.items():
        if roi_frames:
            roi_arrays[roi_name] = np.stack(roi_frames, axis=0)
        else:
            print(f"Warning: No data for {roi_name}")
            roi_arrays[roi_name] = np.zeros((num_frames, 24, 24, 3))
    
    # Create dummy PPG and time deltas (for inference, these will be predicted)
    dummy_ppg = np.zeros(num_frames)
    dummy_time_deltas = np.ones(num_frames) * (1.0 / 30.0)
    dummy_landmarks = np.zeros((num_frames, 468, 2))
    
    # Save to NPZ
    np.savez(
        output_npz_path,
        **roi_arrays,
        ppg_values=dummy_ppg,
        time_deltas=dummy_time_deltas,
        landmarks=dummy_landmarks,
        meta_subject_id=np.array(0),
        meta_condition=np.array('unknown'),
        meta_camera_type=np.array('unknown'),
        meta_view_type=np.array('unknown'),
        meta_chunk_index=np.array(0),
        meta_start_frame=np.array(0),
        meta_end_frame=np.array(num_frames),
        meta_num_frames=np.array(num_frames)
    )
    
    print(f"Saved processed video to: {output_npz_path}")
    return output_npz_path

In [ ]:
def video_inference_pipeline(video_path, model_path, output_dir='./video_inference_results'):
    """End-to-end video inference pipeline."""
    os.makedirs(output_dir, exist_ok=True)
    
    # Step 1: Process video to NPZ format
    video_name = Path(video_path).stem
    npz_path = os.path.join(output_dir, f"{video_name}.npz")
    
    processed_npz = process_video_to_npz_format(video_path, npz_path)
    if processed_npz is None:
        print("Failed to process video")
        return None
    
    # Step 2: Run inference on processed NPZ
    print(f"Running inference on {processed_npz}...")
    pred_ppg, _ = run_single_file_inference(model_path, processed_npz)
    
    # Step 3: Save results
    results = {
        'video_path': video_path,
        'predicted_ppg': pred_ppg,
        'num_frames': len(pred_ppg),
        'timestamp': str(np.datetime64('now'))
    }
    
    results_path = os.path.join(output_dir, f"{video_name}_results.npy")
    np.save(results_path, results)
    
    # Step 4: Plot and save the prediction
    plt.figure(figsize=(15, 5))
    plt.plot(pred_ppg, label='Predicted PPG', color='red', linewidth=2)
    plt.title(f'Predicted PPG from Video: {video_name}', fontsize=14)
    plt.xlabel('Frame')
    plt.ylabel('Normalized PPG')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    plot_path = os.path.join(output_dir, f"{video_name}_ppg.png")
    plt.savefig(plot_path)
    plt.show()
    
    print(f"Results saved to: {results_path}")
    print(f"Plot saved to: {plot_path}")
    
    return pred_ppg

# Data Analysis Section

The following cells handle NPZ file analysis and visualization.

In [ ]:
# Find all NPZ files in the preprocessed data directory
all_npz_files = list(PREPROCESSED_DATA_PATH.rglob('*.npz'))

print(f"Found {len(all_npz_files)} NPZ files in {PREPROCESSED_DATA_PATH}")

# Filter files to only those with exactly EXPECTED_FRAMES
print(f"\nFiltering files to only those with exactly {EXPECTED_FRAMES} frames...")
filtered_files = []
skipped_files = []

for file_path in all_npz_files:
    num_frames = get_num_frames(file_path)
    if num_frames == EXPECTED_FRAMES:
        filtered_files.append(file_path)
    else:
        skipped_files.append((file_path, num_frames))

print(f"Found {len(filtered_files)} files with {EXPECTED_FRAMES} frames")
print(f"Skipped {len(skipped_files)} files with different frame counts")

if skipped_files:
    print("\nSkipped files (first 10):")
    for file_path, num_frames in skipped_files[:10]:
        print(f"  {file_path.name}: {num_frames} frames")

if not filtered_files:
    print("\nNo files found with exactly {EXPECTED_FRAMES} frames. Creating a sample file for demonstration...")
    selected_file = Path('sample_demo.npz')
    
    num_frames = EXPECTED_FRAMES
    time_deltas = np.ones(num_frames) * (1.0 / 30.0)
    sample_data = {
        'roi_full_face': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_forehead': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_left_eye': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_right_eye': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_nose': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_mouth': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_chin': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_right_cheek_50': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_left_cheek_280': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_chin_199': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'ppg_values': np.random.randn(num_frames).astype(np.float32) * 10 + 100,
        'time_deltas': time_deltas.astype(np.float32),
        'landmarks': np.random.rand(num_frames, 468, 2).astype(np.float32),
        'meta_subject_id': np.array(1020),
        'meta_condition': np.array('after'),
        'meta_camera_type': np.array('FullHDwebcam'),
        'meta_view_type': np.array('front'),
        'meta_chunk_index': np.array(0),
        'meta_start_frame': np.array(0),
        'meta_end_frame': np.array(450),
        'meta_num_frames': np.array(450),
        'vital_upper_ap': np.array(113.0),
        'vital_lower_ap': np.array(78.0),
        'vital_saturation': np.array(98.0),
        'vital_hemoglobin': np.array(12.6),
        'vital_glycated_hemoglobin': np.array(5.56),
        'vital_cholesterol': np.array(4.3),
        'vital_respiratory': np.array(19.0),
        'vital_rigidity': np.array(10.79),
        'vital_pulse': np.array(100.0),
        'vital_stress': np.array(4.0),
    }
    
    np.savez(selected_file, **sample_data)
    print(f"Created sample file: {selected_file}")
    filtered_files = [selected_file]

# Select the first file for analysis
selected_file = filtered_files[0]
print(f"\nSelecting file for analysis: {selected_file}")
print(f"Expected frames: {EXPECTED_FRAMES}")

In [ ]:
# Load the selected file
data = load_npz_file(selected_file)

if data is None:
    raise FileNotFoundError(f"Could not load {selected_file}")

# Verify frame count
actual_frames = data['ppg_values'].shape[0]
print(f"\nVerified: File has {actual_frames} frames (expected {EXPECTED_FRAMES})")

# Analyze the structure
analyze_npz_structure(data)

# Extract metadata and ROI names
metadata = extract_metadata(data)
roi_names = extract_roi_names(data)

print("\nMetadata:")
for key, value in metadata.items():
    print(f"  {key}: {value}")

print(f"\nROI Names ({len(roi_names)}):")
for roi in roi_names:
    print(f"  {roi}")

In [ ]:
# Extract PPG data
ppg_values = data['ppg_values']
time_deltas = data['time_deltas']

print(f"PPG Values: shape={ppg_values.shape}, dtype={ppg_values.dtype}")
print(f"Time Deltas: shape={time_deltas.shape}, dtype={time_deltas.dtype}")

# Compute time array with automatic scaling
print("\nComputing time array...")
time_array = compute_time_array(time_deltas, expected_fps=30)
total_time = time_array[-1] - time_array[0]

# Basic statistics
print(f"\nPPG Statistics:")
print(f"  Mean: {ppg_values.mean():.2f}")
print(f"  Std: {ppg_values.std():.2f}")
print(f"  Min: {ppg_values.min():.2f}")
print(f"  Max: {ppg_values.max():.2f}")
print(f"  Range: {ppg_values.max() - ppg_values.min():.2f}")

# Calculate time information
print(f"\nTime Information:")
print(f"  Total duration: {total_time:.2f} seconds")
print(f"  Frame rate: {len(ppg_values) / total_time:.2f} FPS")

In [ ]:
# Plot PPG Signal
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle(f'PPG Analysis - Subject {metadata.get("meta_subject_id", "N/A")} | {metadata.get("meta_condition", "N/A")} | {metadata.get("meta_camera_type", "N/A")}', fontsize=16)

# 1. Raw PPG Signal
ax1 = axes[0, 0]
ax1.plot(time_array, ppg_values, linewidth=1.5, color='red')
ax1.set_title('Raw PPG Signal', fontsize=14)
ax1.set_xlabel('Time (s)', fontsize=12)
ax1.set_ylabel('PPG Value', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_facecolor('#f9f9f9')

# 2. PPG Signal with Mean Line
ax2 = axes[0, 1]
ax2.plot(time_array, ppg_values, linewidth=1.5, color='red', alpha=0.7, label='PPG')
ax2.axhline(y=ppg_values.mean(), color='blue', linestyle='--', linewidth=2, label=f'Mean: {ppg_values.mean():.2f}')
ax2.set_title('PPG Signal with Mean', fontsize=14)
ax2.set_xlabel('Time (s)', fontsize=12)
ax2.set_ylabel('PPG Value', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_facecolor('#f9f9f9')

# 3. PPG Distribution
ax3 = axes[1, 0]
ax3.hist(ppg_values, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
ax3.set_title('PPG Value Distribution', fontsize=14)
ax3.set_xlabel('PPG Value', fontsize=12)
ax3.set_ylabel('Frequency', fontsize=12)
ax3.grid(True, alpha=0.3)
ax3.set_facecolor('#f9f9f9')

# 4. PPG Autocorrelation
ax4 = axes[1, 1]
corr = signal.correlate(ppg_values - ppg_values.mean(), ppg_values - ppg_values.mean(), mode='full')
corr = corr[len(corr)//2:]
lags = np.arange(len(corr)) * (time_array[1] - time_array[0]) if len(time_array) > 1 else np.arange(len(corr))
ax4.plot(lags, corr, linewidth=1.5, color='green')
ax4.set_title('PPG Autocorrelation', fontsize=14)
ax4.set_xlabel('Lag (s)', fontsize=12)
ax4.set_ylabel('Correlation', fontsize=12)
ax4.grid(True, alpha=0.3)
ax4.set_facecolor('#f9f9f9')

plt.tight_layout()
plt.show()

# Training Section

Run the following cells to train the SCNN model on 9 ROIs.

In [ ]:
# Train the model
# Note: This will take a while depending on your dataset size and hardware

print("Starting training...")
model, history = train_pipeline(
    data_dir=str(PREPROCESSED_DATA_PATH),
    epochs=3,           # Start with 3 epochs for testing
    batch_size=4,       # Reduce if you have memory issues
    lr=1e-4
)
print("Training complete!")

# Inference Section

Run inference on NPZ files or video files.

In [ ]:
# Example: Run inference on a single NPZ file
# Replace with your actual model path and NPZ file path

model_path = os.path.join(str(PREPROCESSED_DATA_PATH), "best_scnn_9rois.pth")
sample_npz = str(selected_file)  # Use the file we analyzed earlier

if os.path.exists(model_path):
    print(f"Running inference on {sample_npz}...")
    pred_ppg, gt_ppg = run_single_file_inference(model_path, sample_npz)
    print("Inference complete!")
else:
    print(f"Model not found at {model_path}. Please train the model first or provide a valid model path.")

In [ ]:
# Example: Run inference on a video file
# This requires mediapipe and skvideo to be installed

if VIDEO_DATA_PATH.exists():
    # Get first video file
    video_files = list(VIDEO_DATA_PATH.glob('*.avi')) + list(VIDEO_DATA_PATH.glob('*.mp4'))
    if video_files:
        sample_video = video_files[0]
        print(f"Found video: {sample_video}")
        
        # Run video inference
        if os.path.exists(model_path):
            print(f"Running video inference on {sample_video}...")
            predicted_ppg = video_inference_pipeline(
                video_path=str(sample_video),
                model_path=model_path,
                output_dir='./video_inference_results'
            )
        else:
            print(f"Model not found. Please train first.")
    else:
        print(f"No video files found in {VIDEO_DATA_PATH}")
else:
    print(f"Video data path not found: {VIDEO_DATA_PATH}")

# AVI File Inference Section

Direct inference from AVI video files with synchronized PPG ground truth comparison.

In [ ]:
# ============================================================================
# SCNN_9ROIs Model (Exact Architecture Matching Checkpoint)
# ============================================================================

class SCNN_9ROIs(nn.Module):
    def __init__(self):
        super(SCNN_9ROIs, self).__init__()
        
        # Matches checkpoint shapes: 27 input channels, 3x3x3 spatio-temporal kernels
        self.spatial_temporal_feature_extractor = nn.Sequential(
            nn.Conv3d(in_channels=27, out_channels=32, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(32),
            nn.ELU(),
            nn.Conv3d(in_channels=32, out_channels=64, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(64),
            nn.ELU(),
            nn.AdaptiveAvgPool3d((None, 1, 1))  # Collapses spatial (H, W) to 1x1, keeps temporal depth
        )
        
        # Matches regression head weight shape: torch.Size([1, 64])
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x):
        # Expected input shape: [Batch, 27, Frames, H, W]
        x = self.spatial_temporal_feature_extractor(x)  # Output shape: [Batch, 64, Frames, 1, 1]
        
        # Collapse spatial singleton dimensions -> [Batch, 64, Frames]
        x = x.squeeze(-1).squeeze(-1)
        
        # Permute to frame-wise linear configuration -> [Batch, Frames, 64]
        x = x.permute(0, 2, 1)
        
        # Apply the frame-wise regression head -> [Batch, Frames, 1]
        out = self.regression_head(x)
        
        # Return standard flat layout -> [Batch, Frames]
        return out.squeeze(-1)

In [ ]:
# ============================================================================
# AVI Video Processing Configuration
# ============================================================================

# Configuration for video chunking
CHUNK_SIZE = 450  # Number of frames per chunk
OVERLAP_SIZE = 0  # No overlap for simplicity

# ROI key mapping for AVI processing
AVI_ROI_KEYS = ['forehead', 'left_eye', 'right_eye', 'nose', 'mouth', 'chin', 
               'right_cheek_50', 'left_cheek_280', 'chin_199']

# Map AVI ROI keys to NPZ ROI keys
AVI_TO_NPZ_ROI = {
    'forehead': 'roi_forehead',
    'left_eye': 'roi_left_eye',
    'right_eye': 'roi_right_eye',
    'nose': 'roi_nose',
    'mouth': 'roi_mouth',
    'chin': 'roi_chin',
    'right_cheek_50': 'roi_right_cheek_50',
    'left_cheek_280': 'roi_left_cheek_280',
    'chin_199': 'roi_chin_199'
}

In [ ]:
# ============================================================================
# AVI Video Chunking and Processing
# ============================================================================

def chunk_video_for_inference(video_path, chunk_size=CHUNK_SIZE, overlap_size=OVERLAP_SIZE):
    """Chunk video into manageable pieces for inference."""
    try:
        import skvideo.io
        videodata = skvideo.io.vread(video_path)
        
        if len(videodata) == 0:
            return []
        
        # Split into chunks
        chunks = []
        for i in range(0, len(videodata), chunk_size - overlap_size):
            end_idx = min(i + chunk_size, len(videodata))
            chunk = videodata[i:end_idx]
            
            # Pad if necessary
            if len(chunk) < chunk_size:
                padding = chunk_size - len(chunk)
                last_frame = chunk[-1:].repeat(padding, axis=0)
                chunk = np.concatenate([chunk, last_frame], axis=0)
            
            chunks.append(chunk)
        
        return chunks
    except Exception as e:
        print(f"Error chunking video: {e}")
        return []

In [ ]:
def process_inference_chunk(chunk, detector, video_path=None):
    """Process a video chunk to extract ROIs for inference."""
    try:
        import mediapipe as mp
        
        roi_data = {roi: [] for roi in AVI_ROI_KEYS}
        all_landmarks = []
        
        for frame in chunk:
            # Convert to RGB if needed
            rgb_frame = frame.copy()
            if len(rgb_frame.shape) == 2:
                rgb_frame = np.stack([rgb_frame] * 3, axis=-1)
            elif rgb_frame.shape[2] == 4:
                rgb_frame = rgb_frame[:, :, :3]
            
            # Detect face landmarks
            results = detector.process(rgb_frame)
            
            if results.multi_face_landmarks:
                landmarks = results.multi_face_landmarks[0].landmark
                height, width = frame.shape[:2]
                landmark_points = np.array([
                    (lmk.x * width, lmk.y * height) 
                    for lmk in landmarks
                ])
                all_landmarks.append(landmark_points)
                
                # Extract each ROI
                for avi_roi_key in AVI_ROI_KEYS:
                    npz_roi_key = AVI_TO_NPZ_ROI[avi_roi_key]
                    if npz_roi_key in ROI_DEFINITIONS:
                        roi_indices = ROI_DEFINITIONS[npz_roi_key]
                        roi = extract_roi_from_frame(frame, landmark_points, roi_indices, size=24)
                        if roi is not None:
                            roi_data[avi_roi_key].append(roi)
            else:
                # No face detected - use previous landmarks
                if all_landmarks:
                    landmark_points = all_landmarks[-1]
                    for avi_roi_key in AVI_ROI_KEYS:
                        npz_roi_key = AVI_TO_NPZ_ROI[avi_roi_key]
                        if npz_roi_key in ROI_DEFINITIONS:
                            roi_indices = ROI_DEFINITIONS[npz_roi_key]
                            roi = extract_roi_from_frame(frame, landmark_points, roi_indices, size=24)
                            if roi is not None:
                                roi_data[avi_roi_key].append(roi)
        
        # Stack ROIs into arrays
        for key in roi_data:
            if roi_data[key]:
                roi_data[key] = np.stack(roi_data[key], axis=0)
            else:
                roi_data[key] = np.zeros((len(chunk), 24, 24, 3))
        
        return {'roi_data': roi_data, 'landmarks': all_landmarks}
    except Exception as e:
        print(f"Error processing chunk: {e}")
        return None

In [ ]:
# ============================================================================
# AVI File Inference with Ground Truth Comparison
# ============================================================================

def run_avi_inference(video_path, ppg_sync_path=None, model_path='best_scnn_9rois.pth'):
    """Run inference on AVI file with optional ground truth comparison."""
    
    # Initialize device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Target execution hardware: {device}")
    
    # Initialize model
    model = SCNN_9ROIs()
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"Loaded model from {model_path}")
    else:
        print(f"Warning: Model not found at {model_path}. Using random weights.")
    
    model.to(device)
    model.eval()
    
    # Initialize MediaPipe detector
    try:
        import mediapipe as mp
        detector = mp.solutions.face_mesh.FaceMesh(
            static_image_mode=False,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5
        )
    except ImportError:
        print("mediapipe not available. Install with: pip install mediapipe")
        return None
    
    # Chunk video
    video_chunks = chunk_video_for_inference(video_path, chunk_size=CHUNK_SIZE, overlap_size=OVERLAP_SIZE)
    if not video_chunks:
        print("Video extraction yielded 0 active frames.")
        return None
    
    # Process first chunk
    sample_chunk = video_chunks[0]
    processed_data = process_inference_chunk(sample_chunk, detector, video_path=video_path)
    
    if processed_data is None:
        print("Failed to process video chunk")
        return None
    
    roi_dict = processed_data['roi_data']
    
    # Extract and stack ROIs
    rois_list = [roi_dict[key] for key in AVI_ROI_KEYS]
    stacked_rois = np.concatenate(rois_list, axis=-1)  # (450, 24, 24, 27)
    stacked_rois = np.transpose(stacked_rois, (3, 0, 1, 2))  # (27, 450, 24, 24)
    
    # Create tensor
    input_tensor = torch.from_numpy(stacked_rois).float().unsqueeze(0)
    if input_tensor.max() > 1.0:
        input_tensor = input_tensor / 255.0
    
    input_tensor = input_tensor.to(device)
    
    # Run inference
    with torch.no_grad():
        predictions = model(input_tensor)
    
    pred_signal = predictions.squeeze(0).cpu().numpy()
    print(f"Predicted rPPG signal frame depth: {pred_signal.shape[0]} points.")
    
    # Plot prediction
    plt.figure(figsize=(15, 5))
    timeline = np.arange(len(pred_signal)) / 30.0
    plt.plot(timeline, pred_signal, label='SCNN Prediction', color='#e91e63', linewidth=2, linestyle='--')
    plt.title(f"AVI Video Inference: {Path(video_path).name}", fontsize=12, fontweight='bold')
    plt.xlabel('Timeline Duration (Seconds)')
    plt.ylabel('Predicted PPG')
    plt.legend(loc='upper right')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.tight_layout()
    plt.show()
    
    # Compare with ground truth if available
    if ppg_sync_path and os.path.exists(ppg_sync_path):
        raw_gt = np.loadtxt(ppg_sync_path).astype(np.float32)
        gt_signal = raw_gt[0:CHUNK_SIZE]
        
        if len(gt_signal) < len(pred_signal):
            gt_signal = np.pad(gt_signal, (0, len(pred_signal) - len(gt_signal)), 'edge')
        
        # Standardize
        pred_signal_norm = (pred_signal - np.mean(pred_signal)) / (np.std(pred_signal) + 1e-6)
        gt_signal_norm = (gt_signal - np.mean(gt_signal)) / (np.std(gt_signal) + 1e-6)
        
        # Force to 1D and match lengths
        pred_flat = pred_signal_norm.ravel()
        gt_flat = gt_signal_norm.ravel()
        min_len = min(len(pred_flat), len(gt_flat))
        pred_flat = pred_flat[:min_len]
        gt_flat = gt_flat[:min_len]
        
        # Plot comparison
        plt.figure(figsize=(15, 5))
        timeline = np.arange(min_len) / 30.0
        plt.plot(timeline, gt_flat, label='Ground Truth BVP/PPG', color='#009688', linewidth=2)
        plt.plot(timeline, pred_flat, label='SCNN Prediction', color='#e91e63', linewidth=2, linestyle='--')
        plt.title(f"SCNN 9-ROI Inference (Subject: {Path(video_path).stem})", fontsize=12, fontweight='bold')
        plt.xlabel('Timeline Duration (Seconds)')
        plt.ylabel('Standardized Amplitude (Z-Score)')
        plt.legend(loc='upper right')
        plt.grid(True, linestyle=':', alpha=0.5)
        plt.tight_layout()
        plt.show()
        
        # Compute metrics
        pearson_corr = np.corrcoef(pred_flat, gt_flat)[0, 1]
        print(f"Pearson Correlation (r): {pearson_corr:.4f}")
    else:
        print(f"[Note]: No PPG sync file provided or found at: {ppg_sync_path}")
    
    return pred_signal

# Summary

This notebook provides:
1. **Data Analysis**: Load and analyze NPZ files with 450 frames
2. **PPG Visualization**: Plot PPG signals with various analyses
3. **Model Training**: Train SCNN on 9 ROIs with temporal consistency loss
4. **Inference**: Run inference on NPZ files or video files

## Requirements

For full functionality, install:
```bash
pip install torch numpy matplotlib seaborn scikit-video mediapipe scikit-image
```